# TAREA 2: REGRESIÓN

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Como queremos predecir nota_media_2sem, ahora nuestro target será esa variable

Queremos predecir una nota, una variable continua: estamos ante un problema de regresión

Por lo tanto, no podremos usar variables del 2º semestre (porque estamos prediciendo esa nota). Así que todas las variables que contengan 2sem las eliminaremos de nuestra X

In [2]:
df = pd.read_csv("rendimiento_estudiantes.csv", sep=";")

Además, vamos a generar dos escenarios:

    - A: No tenemos en cuenta ni los datos del 1er ni el 2º semestre

    - B: Solo dejamos de tener en cuenta los datos del 2º semestre

Y compararemos el rendimiento de los modelos según los datos de los que partimos

In [3]:
y = df["nota_media_2sem"]

cols_1sem = [col for col in df.columns if "1sem" in col]
cols_2sem = [col for col in df.columns if "2sem" in col]

X_A = df.drop(cols_1sem + cols_2sem, axis=1)
X_B = df.drop(cols_2sem, axis=1)

Hacemos one-hot-encoding para transformar las variables categóricas en variables numéricas binarias (0 ó 1)

In [4]:
X_A = pd.get_dummies(X_A)
X_B = pd.get_dummies(X_B)

# Para alinear las columnas de ambos
X_A = X_A.reindex(columns=X_B.columns, fill_value=0)

Por último, dividimos nuestros datos en los conjuntos de train y test

In [5]:
X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(X_A, y, test_size=0.2, random_state=42)

X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(X_B, y, test_size=0.2, random_state=42)

Aplicaremos el modelo de regresión lineal porque nos servirá como referencia para saber si la relación es lineal y es simple o no

In [6]:
modelo_A_lin = LinearRegression()
modelo_A_lin.fit(X_train_A, y_train_A)

y_A_pred_lin = modelo_A_lin.predict(X_test_A)


modelo_B_lin = LinearRegression()
modelo_B_lin.fit(X_train_B, y_train_B)

y_B_pred_lin = modelo_B_lin.predict(X_test_B)

También aplicaremos el modelo de Random Forest porque es más complejo y robusto y capturará las relaciones no lineales (algo que claramente no consigue hacer el modelo lineal)

In [7]:
modelo_A_rf = RandomForestRegressor(random_state=42)
modelo_A_rf.fit(X_train_A, y_train_A)

y_A_pred_rf = modelo_A_rf.predict(X_test_A)


modelo_B_rf = RandomForestRegressor(random_state=42)
modelo_B_rf.fit(X_train_B, y_train_B)

y_B_pred_rf = modelo_B_rf.predict(X_test_B)

In [8]:
def evaluar(y_test, y_pred, nombre):
    print(f"\nEvaluación de {nombre}")
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print("RMSE:", rmse)
    print("MAE:", mae)
    print("R2:", r2)

print("CASO A: No tenemos en cuenta ni los datos del 1er ni el 2º semestre")
evaluar(y_test_A, y_A_pred_lin, "Regresión Lineal")
evaluar(y_test_A, y_A_pred_rf, "Random Forest")

print("\nCASO B: Tenemos en cuenta todo menos el segundo semestre")
evaluar(y_test_B, y_B_pred_lin, "Regresión Lineal")
evaluar(y_test_B, y_B_pred_rf, "Random Forest")

CASO A: No tenemos en cuenta ni los datos del 1er ni el 2º semestre

Evaluación de Regresión Lineal
RMSE: 3004803986.410577
MAE: 216359954.73237827
R2: -3.3348842838887085e+17

Evaluación de Random Forest
RMSE: 4.001943632532475
MAE: 2.795094287653648
R2: 0.40845142822380365

CASO B: Tenemos en cuenta todo menos el segundo semestre

Evaluación de Regresión Lineal
RMSE: 1313497470.4279387
MAE: 92724026.68667002
R2: -6.372457637131892e+16

Evaluación de Random Forest
RMSE: 2.6353422485307227
MAE: 1.4038975705198384
R2: 0.7434792361173092


Debido al pésimo rendimiento del modelo de regresión lineal en ambos casos, probamos a aplicar la regularización de Ridge y observamos una clarísima mejora (incluso puede llegar a proporcionar un rendimiento superior al que nos brinda Random Forest).

In [9]:
modelo_ridge_A = Ridge(alpha=1.0)
modelo_ridge_A.fit(X_train_A, y_train_A)

y_pred_ridge_A = modelo_ridge_A.predict(X_test_A)


modelo_ridge_B = Ridge(alpha=1.0)
modelo_ridge_B.fit(X_train_B, y_train_B)

y_pred_ridge_B = modelo_ridge_B.predict(X_test_B)


evaluar(y_test_A, y_pred_ridge_A, "Ridge en caso A")
evaluar(y_test_B, y_pred_ridge_B, "Ridge en caso B")


Evaluación de Ridge en caso A
RMSE: 4.0427326344692505
MAE: 2.9999800120988556
R2: 0.3963314978132175

Evaluación de Ridge en caso B
RMSE: 2.6092073051994897
MAE: 1.641093886165739
R2: 0.7485418899354217
